# Learning Output Parser
**The way to format output of a LLM model**

In [40]:
import subprocess
subprocess.Popen(["ollama","serve"],stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

prompt = "What is the capital of France??"

from langchain_ollama import ChatOllama
llm = ChatOllama(model="gemma4:31b-cloud",temperature=0.7)
ans = llm.invoke(prompt)
print(ans.content)

The capital of France is **Paris**.


## String Output Parser

In [41]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

#Creating a template
template = PromptTemplate.from_template("What is the capital of {country}??")
chat = template.invoke({"country":"france"})

response = llm.invoke(chat)
print(response.content)


The capital of France is **Paris**.


## With Output parser

In [42]:
parser = StrOutputParser()
final_result = parser.invoke(response)
print(final_result)

The capital of France is **Paris**.


## With Chain
**Means using utlities**

In [43]:
parser = StrOutputParser()
template = PromptTemplate.from_template("What is the capital of {country}??")
#chat = template.invoke({"country":"france"})

chain = template | llm | parser
response = chain.invoke({"country":"France"})
print(response)

The capital of France is **Paris**.


In [51]:
parser = StrOutputParser()
llm1 = ChatOllama(model="gemma4:31b-cloud",temperature=0.7)
template = PromptTemplate.from_template("What is the C program for swapping {number} numbers")
llm2 = ChatOllama(model="llama3.2:latest",temperature=0.5)
chain = template | llm1 | parser| llm2 | parser
response = chain.invoke({"number":"3"})
print(response)

The code provided is a C program that performs a circular swap of three integers without using a temporary variable. It uses addition and subtraction to achieve this.

Here's a breakdown of the code:

```c
#include <stdio.h>

int main() {
    int a, b, c;

    // Input three numbers
    printf("Enter first number (a): ");
    scanf("%d", &a);
    printf("Enter second number (b): ");
    scanf("%d", &b);
    printf("Enter third number (c): ");
    scanf("%d", &c);

    printf("\nBefore swapping: a = %d, b = %d, c = %d\n", a, b, c);

    // Circular swap without temp
    a = a + b + c; // Store sum of all three in a
    b = a - (b + c); // Assign original a to b by subtracting the sum from a
    c = a - (b + c); // Assign original b to c by subtracting the sum from a
    a = a - (b + c); // Assign original c to a by subtracting the sum from a

    printf("After swapping:  a = %d, b = %d, c = %d\n", a, b, c);

    return 0;
}
```

This code works because of the properties of arithmetic op

## Json output parser

In [52]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
template = PromptTemplate.from_template("""
Extract the following information written in text to JSON format:
-name
-price
-category
-features list
text: {text}
format-instructions:{format-instructions}
""",partial_variables={"format-instructions":json_parser.get_format_instructions()})
chain = template | llm | json_parser
result = chain.invoke({"text": "The Galactic Keyboard costs $150. It's a gaming peripheral featuring RGB lights and mechanical switches."})

print(result)

{'name': 'Galactic Keyboard', 'price': '$150', 'category': 'gaming peripheral', 'features': ['RGB lights', 'mechanical switches']}


### Extract cpp code

In [53]:
template = PromptTemplate.from_template("""
Extract code metadata to JSON:
- language
- libraries_used (list)
- variables_defined (list)
- purpose
text: {text}
format-instructions:{format-instructions}
""", partial_variables={"format-instructions":json_parser.get_format_instructions()})

chain = template | llm | json_parser
result = chain.invoke({"text":"""Of course, here's an updated version of the program that swaps three integers using the `swap` function from the `<functional>` header:

```c
#include <iostream> // required for std::cout and std::endl
#include <functional> // required for swap() function

int main() {
    int a = 10, b = 20, c = 30;
    
    // Calling the `swap` function with the `std::swap()` operator to swap values
    std::swap(a, b);
    std::swap(b, c);
    
    // Swapping the values of a, b, and c using the `swap()` function from `<functional>`
    std::swap(a, c);
    std::swap(b, a);
    
    // Printing the swapped values of a, b, and c using `std::cout`
    std::cout << "After swapping: a = " << a << ", b = " << b << ", c = " << c << "\n";
    
    return 0;
}
```

This program uses the `swap()` function from the `<functional>` header to swap the values of three integers, `a`, `b`, and `c`. It then prints the swapped values using the `std::cout` stream.
"""})

print(result)

{'language': 'c++', 'libraries_used': ['iostream', 'functional'], 'variables_defined': ['a', 'b', 'c'], 'purpose': 'Swap the values of three integers using the std::swap function and print the results.'}


## Pydantic Output Parser

In [54]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Optional, List

# Creating output type for llm model
class Movie(BaseModel):
    title:str=Field(...,description="Name of the movie")
    release_year:int=Field(..., description="Year the movie was released")
    genres:List[str] = Field(default_factory=list, description="List of genres the movie belongs to")
    rating:Optional[float]=Field(None,description="Average rating of the movie on scale 1 to 10")
    box_office:Optional[float]=Field(None,description="Worldwide box office in millions USD if known")

#Setting up pydantic parser so llm follows the rules
pydantic_parser = PydanticOutputParser(pydantic_object=Movie)
#Creating templates / framework for the output type 
template = PromptTemplate.from_template("""
Extract the movie information from following text:
{text}\
{format_instructions}""",
partial_variables={"format_instructions":pydantic_parser.get_format_instructions()})


## Giving a text as input
text = """
The Dark Knight was released in 2008, directed by Christopher Nolan.
It stars Christian Bale as Batman and Heath Ledger as the Joker.
The film is an action thriller with elements of crime drama.
It holds a rating of 9.0 on IMDb and earned over $1 billion at the box office.
"""

chain = template | llm | pydantic_parser
try:
    result = chain.invoke({"text":text})
    print(result)
except Exception as e:
    print(f"Error occures at {e}")

title='The Dark Knight' release_year=2008 genres=['action thriller', 'crime drama'] rating=9.0 box_office=1000.0
